# Arrays and indexing

> Compact arrays, leading-axis broadcasting, names, keys and Python interoperability.

In [ ]:
import rustymath as rm

Rustymath arrays own compact, row-major data. Python operations run eagerly and materialize their results. Slicing returns compact copies rather than aliased views. Float display rounding depends on the dtype and does not change the stored values returned by `.tolist()`. For expression fusion and richer array notation, use [BasedPL](https://answerdotai.github.io/basedpl/).

## Construction and dtypes

`array` copies Python values or a buffer exporter into an owned array. `dtype` selects its item type. Fill constructors take a shape; their `_like` forms retain another array's shape, keys and names:

In [ ]:
temperatures = rm.array([[18., 20, 22], [16, 19, 23]], dtype='f32')
temperatures, rm.zeros_like(temperatures), rm.full((2, 3), 1)

(Array([[18.0, 20.0, 22.0], [16.0, 19.0, 23.0]], dtype=f32),
 Array([[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], dtype=f32),
 Array([[1, 1, 1], [1, 1, 1]], dtype=i64))

`arange` generates values before its stop. `linspace` includes both endpoints by default. `logspace` generates evenly spaced exponents in the chosen base:

In [ ]:
rm.arange(0, 6, 2), rm.linspace(0, 1, 5), rm.logspace(0, 3, 4)

(Array([0, 2, 4], dtype=i64),
 Array([0.0, 0.25, 0.5, 0.75, 1.0], dtype=f64),
 Array([1.0, 10.0, 100.0, 1000.0], dtype=f64))

Dtype helpers use the same supported item types as arrays. `finfo` reports float limits and spacing at 1; `iinfo` reports integer limits. `result_type` gives the shared promotion rule:

In [ ]:
rm.finfo('f32').eps, rm.iinfo('i16').max, rm.result_type('i16', 'f16'), rm.issubdtype('f32', 'floating')

(1.1920928955078125e-07, 32767, 'f32', True)

## Broadcasting and conversion

Each function takes numbers, nested lists, rustymath arrays, NumPy arrays, or any other object that exports a buffer. It returns a `rustymath.Array`, or a Python number when every argument is a Python number. Arguments of different shapes broadcast by aligning their leading axes, where NumPy aligns trailing axes. An axis of length 1 stretches to match. `broadcast_to` makes this explicit and returns a compact copy. Here a vector of two items adds one number to each row of a 2 by 3 matrix:

In [ ]:
rm.add([[1, 2, 3], [4, 5, 6]], [10, 20])

Array([[11, 12, 13], [24, 25, 26]], dtype=i64)

Arrays of different dtypes promote as in NumPy: the result takes the higher kind, wide enough to hold both, so an `i16` array beside an `f16` array gives `f32`. Characters don't mix with numbers. An integer result that doesn't fit its dtype raises `OverflowError`. A Python number or list follows NEP 50, as in NumPy: it takes the dtype of the array beside it, unless its kind is higher, so a float beside an integer array gives `f64`. A function that computes only on floats, such as `exp`, converts Booleans and integers to the narrowest float that holds them exactly, but at least `f32`, which computes faster than `f16`. So Booleans, `u8` and `i16` give `f32`, and wider integers give `f64`:

In [ ]:
i16 = rm.array([1, 2]).astype('i16')
rm.add(i16, 3), rm.add(i16, 0.5), rm.add(i16, rm.array([0.5, 1]).astype('f16')), rm.exp(i16), rm.exp([1, 2])

(Array([4, 5], dtype=i16),
 Array([1.5, 2.5], dtype=f64),
 Array([1.5, 3.0], dtype=f32),
 Array([2.71828, 7.38906], dtype=f32),
 Array([2.7182818, 7.3890561], dtype=f64))

A `rustymath.Array` has arithmetic, bitwise and comparison operators, `@`, and the methods `.sum()`, `.prod()`, `.max()`, `.min()`, `.all()`, `.any()`, `.cumsum()`, `.cumprod()`, `.reshape()`, `.astype()` and `.tolist()`. With no `axis`, a reduction reduces the whole array to a number, and a scan runs over the flattened array:

In [ ]:
m = rm.array([[1, 2], [3, 4]])
(m * 10).sum(), m.sum(axis=0), m.cumsum()

(100, Array([4, 6], dtype=i64), Array([1, 3, 6, 10], dtype=i64))

## Selecting and reshaping

Brackets select each axis independently. Two lists select their Cartesian product, not paired positions. Integers remove an axis; lists and one-dimensional Boolean masks keep it. Negative positions count from the end, and slices take Python's start, stop and step syntax:

In [ ]:
grid = rm.array([[10, 20, 30], [40, 50, 60]])
grid[[1, 0], [2, 0]], grid[:, ::-1], grid[:, [True, False, True]]

(Array([[60, 40], [30, 10]], dtype=i64),
 Array([[30, 20, 10], [60, 50, 40]], dtype=i64),
 Array([[10, 30], [40, 60]], dtype=i64))

`...` fills in omitted axes and `None` inserts an axis of length one. `reshape`, transpose and joining operations return compact arrays too:

In [ ]:
grid[None, ...].shape, grid.T, rm.concatenate([grid, grid], axis=0).shape

((1, 2, 3), Array([[10, 40], [20, 50], [30, 60]], dtype=i64), (4, 3))

`eye` constructs an identity or rectangular diagonal matrix. `diag` constructs a matrix from a vector, or extracts a matrix's diagonal. `diagonal` works on any two axes, moving the diagonal to the last axis. Its axes default to the last two, the axes that the matrix functions take, where NumPy defaults to the first two. `tril` and `triu` keep one triangle of each matrix. All return compact copies.

In [ ]:
rm.eye(3), rm.diagonal(grid), rm.tril(grid)

(Array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]], dtype=f64),
 Array([10, 50], dtype=i64),
 Array([[10, 0, 0], [40, 50, 0]], dtype=i64))

A selection returns a Python scalar when every axis is removed, and a compact copy otherwise. Assignment uses the same selectors and leading-axis broadcasting. It preserves the array's dtype and shape. For repeated unkeyed targets, the last assigned value wins:

In [ ]:
selected = grid[:, 1]
grid[:, 1] = [200, 500]
selected, grid

(Array([20, 50], dtype=i64), Array([[10, 200, 30], [40, 500, 60]], dtype=i64))

## Names and keys

A dictionary becomes a vector whose axis has the dictionary's keys. Functions align keyed axes by key:

In [ ]:
rm.add({'a': 1.5, 'b': 2.5}, {'b': 10.0, 'a': 20.0})

Array([21.5, 12.5], dtype=f64, keys=(('a', 'b'),))

Keys select positions too. Reordering a keyed axis preserves its keys. Selecting the same key twice raises an error because an axis's keys must be unique:

In [ ]:
keyed = rm.array({'a': [1, 2], 'b': [3, 4]})
keyed[['b', 'a']]

Array([[3, 4], [1, 2]], dtype=i64, keys=(('b', 'a'), None))

Assigning a tuple to `.names` names the axes, with one name or `None` for each axis. `axis` then takes a name wherever it takes a position:

In [ ]:
s = rm.array([[10, 20, 30], [40, 50, 60]])
s.names = ('city', 'month')
s.sum(axis='month')

Array([60, 150], dtype=i64, names=('city',))

Index arrays with axis names are the exception to independent selection. Two of them with the same name pair, as named axes pair in arithmetic, and read one position along both: `m[i, i]` is the diagonal. A named index array also reads the fill value, 0 for numbers, outside its axis, and a negative position in it never counts from the end:

In [ ]:
m = rm.array([[0, 1, 2], [3, 4, 5], [6, 7, 8]])
i = rm.arange(3)
i.names = ('i',)
k = rm.array([1, 2, 3])
k.names = ('i',)
m[i, i], m[k, 0]

## Interoperability and threads

Rustymath arrays export Python's buffer protocol. NumPy can read them through `np.asarray` without copying. Rustymath accepts NumPy arrays and other buffer exporters too. Its Python names follow familiar NumPy or SciPy spellings where they fit the shared core. Broadcasting and indexing follow rustymath's rules.

Here `exp` keeps its argument's 32-bit width:

In [ ]:
import numpy as np

In [ ]:
x = np.linspace(0, 2, 5, dtype=np.float32)
rm.exp(x)

Array([1.0, 1.64872, 2.71828, 4.48169, 7.38906], dtype=f32)

Numeric buffer exports are writable. Boolean and character buffers export read-only, but ordinary rustymath bracket assignment supports both safely.

`rm.set_threads(n)` limits parallel calls on the calling thread and returns its previous setting. Use 0 for the default or 1 for serial calls. Results need not agree in their last bits across thread counts.